# 06 - Prediccion final

**TP2 - Modulo 2 - Clasificacion `smoking`**

Generamos las predicciones sobre los 5.692 registros sin etiquetar y exportamos el archivo de entrega. Esta notebook no reentrena nada: carga el modelo guardado en el notebook 05 y lo aplica directamente.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import joblib

import utils
from utils import PRED_XLSX, DATA_PROCESSED, MODELS, ID_COL, TARGET

# 1. Carga del dataset sin etiquetar con el mismo saneo que en entrenamiento.
df_pred = utils.cargar_y_sanear(PRED_XLSX)
print('Dataset de prediccion:', df_pred.shape)

Dataset de prediccion: (5692, 34)


## Preparacion

Guardamos los `id` antes de transformar porque los necesitamos en la salida. El pipeline ya sabe que columnas usar.

In [2]:
ids = df_pred[ID_COL].values
X_pred, _ = utils.features_target(df_pred)   # quita id/oral (y target si estuviera)
print('X_pred:', X_pred.shape)

X_pred: (5692, 32)


## Carga del modelo

In [3]:
modelo = joblib.load(MODELS / 'best_model.joblib')
with open(MODELS / 'decision_threshold.json') as f:
    cfg = json.load(f)
threshold = cfg['threshold']
print('Modelo:', cfg['modelo'], '| threshold:', round(threshold, 4))

Modelo: XGBoost | threshold: 0.4248


## Prediccion

In [4]:
proba = modelo.predict_proba(X_pred)[:, 1]
pred = (proba >= threshold).astype(int)

salida = pd.DataFrame({ID_COL: ids, 'smoking_prediction': pred})
print(salida['smoking_prediction'].value_counts(normalize=True).round(4))
salida.head()

smoking_prediction
0    0.532
1    0.468
Name: proportion, dtype: float64


,id,smoking_prediction
0,27358,1
1,27364,1
2,27368,1
3,27378,0
4,27381,1


## Verificaciones

Antes de exportar chequeamos que el formato sea el correcto.

In [5]:
assert list(salida.columns) == [ID_COL, 'smoking_prediction'], 'Columnas incorrectas'
assert salida['smoking_prediction'].isin([0, 1]).all(), 'Hay valores fuera de {0,1}'
assert salida[ID_COL].notna().all() and salida['smoking_prediction'].notna().all(), 'Hay nulos'
assert len(salida) == len(df_pred), 'Cantidad de filas no coincide'
print('Verificaciones OK -', len(salida), 'filas')

Verificaciones OK - 5692 filas


## Exportacion

In [6]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
out_path = DATA_PROCESSED / 'predicciones_smoking.csv'
salida.to_csv(out_path, index=False)
print('Predicciones exportadas en:', out_path)

Predicciones exportadas en: /home/mvieira/Documentos/diplomatura/entregas/tp2/data/processed/predicciones_smoking.csv
